# 04 · Inventory Twin (Step 4)

The **Inventory / Material Flow twin** — the connective tissue of the supply chain: what stock
exists, where, how it moves, and how long it lasts under current and forecast demand.

**Owns:** on-hand stock, days-of-cover, stockout/overstock risk, consumption velocity,
projected balance. **Consumes** (does not redefine) the Demand twin's demand rate + forecast.

**Inputs:** `gold.inventorysnapshots` (rich pre-computed daily stock), `monitoring.demand_twin_current`.
**Outputs (monitoring):** `inventory_twin_current`, `inventory_twin_metrics_timeseries`.

Grain: material × plant (aggregated across storage location & batch). Idempotent overwrite.

## Config

In [ ]:
dbutils.widgets.text("source_catalog", "mickmickstableaws_catalog", "Source catalog")
dbutils.widgets.text("source_schema", "gold", "Source schema")
dbutils.widgets.text("target_catalog", "mickmickstableaws_catalog", "Target catalog")
dbutils.widgets.text("monitoring_schema", "monitoring", "Monitoring schema")
dbutils.widgets.text("low_cover_days", "14", "Days-of-cover below this = low/at-risk")
dbutils.widgets.text("overstock_days", "90", "Days-of-cover above this = overstock")

import uuid
from pyspark.sql import functions as F

SRC = f"{dbutils.widgets.get('source_catalog').strip()}.{dbutils.widgets.get('source_schema').strip()}"
FQ_MON = f"{dbutils.widgets.get('target_catalog').strip()}.{dbutils.widgets.get('monitoring_schema').strip()}"
LOW = int(dbutils.widgets.get("low_cover_days"))
OVER = int(dbutils.widgets.get("overstock_days"))
REFRESH_ID = str(uuid.uuid4())

SNAP = f"{SRC}.inventorysnapshots"
DEMAND = f"{FQ_MON}.demand_twin_current"
CUR_OUT = f"{FQ_MON}.inventory_twin_current"
TS_OUT  = f"{FQ_MON}.inventory_twin_metrics_timeseries"
print(f"Source: {SNAP}\nConsumes: {DEMAND}\nOutputs: {CUR_OUT} , {TS_OUT}")

## 1. Build `inventory_twin_current`
Latest snapshot per material × plant + demand-driven days-of-cover and risk (using the Demand twin).

In [ ]:
current = spark.sql(f"""
WITH latest AS (SELECT MAX(SnapshotDate) AS d FROM {SNAP}),
cur AS (
  SELECT s.Client_MANDT AS mandt, s.MaterialNumber_MATNR AS material, s.Plant_WERKS AS plant,
    MAX(s.MaterialText_MAKTX) AS material_text, MAX(s.MaterialGroup_MATKL) AS material_group,
    MAX(s.UoM_MEINS) AS uom,
    SUM(s.UnrestrictedStockQuantity) AS on_hand_qty,
    SUM(s.TrailingDailyConsumptionRate) AS daily_consumption_rate,
    MAX(s.SnapshotDate) AS snapshot_date
  FROM {SNAP} s JOIN latest l ON s.SnapshotDate = l.d
  GROUP BY 1,2,3
),
dmd AS (
  SELECT mandt, material, plant,
    SUM(demand_13w_avg) AS demand_13w_avg_wk,
    SUM(next_period_forecast_qty) AS next_period_forecast_qty,
    SUM(horizon_total_forecast_qty) AS horizon_total_forecast_qty
  FROM {DEMAND} GROUP BY 1,2,3
)
SELECT c.mandt, c.material, c.plant, c.material_text, c.material_group, c.uom, c.snapshot_date,
  c.on_hand_qty, c.daily_consumption_rate,
  CASE WHEN c.daily_consumption_rate > 0 THEN c.on_hand_qty / c.daily_consumption_rate END AS days_of_cover_actual,
  d.demand_13w_avg_wk, d.next_period_forecast_qty, d.horizon_total_forecast_qty,
  (d.demand_13w_avg_wk / 7.0) AS daily_demand,
  CASE WHEN d.demand_13w_avg_wk > 0 THEN c.on_hand_qty / (d.demand_13w_avg_wk / 7.0) END AS days_of_cover_demand,
  CASE WHEN d.horizon_total_forecast_qty > 0 THEN c.on_hand_qty / d.horizon_total_forecast_qty END AS horizon_coverage_ratio,
  (c.on_hand_qty <= 0) AS is_stockout,
  (d.demand_13w_avg_wk > 0 AND c.on_hand_qty / (d.demand_13w_avg_wk / 7.0) < {LOW}) AS is_low_cover,
  (d.demand_13w_avg_wk > 0 AND c.on_hand_qty / (d.demand_13w_avg_wk / 7.0) > {OVER}) AS is_overstock,
  CASE WHEN d.demand_13w_avg_wk > 0 AND c.on_hand_qty > 0
       THEN date_add(c.snapshot_date, CAST(c.on_hand_qty / (d.demand_13w_avg_wk / 7.0) AS INT)) END AS projected_stockout_date
FROM cur c LEFT JOIN dmd d USING (mandt, material, plant)
""")

current = (current
    .withColumn("as_of_date", F.col("snapshot_date"))
    .withColumn("refresh_id", F.lit(REFRESH_ID))
    .withColumn("generated_at", F.current_timestamp()))
(current.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(CUR_OUT))
spark.sql(f"COMMENT ON TABLE {CUR_OUT} IS 'Inventory twin current state per material x plant: on-hand, days-of-cover (actual + demand-driven), stockout/overstock risk, projected stockout.'")
print(f"Wrote {current.count():,} material-plant positions to {CUR_OUT}")

## 2. Build `inventory_twin_metrics_timeseries`
Stock position over time per material × plant × snapshot date (on-hand, receipts, issues, cover).

In [ ]:
timeseries = spark.sql(f"""
SELECT Client_MANDT AS mandt, MaterialNumber_MATNR AS material, Plant_WERKS AS plant,
       SnapshotDate AS period_start,
  SUM(UnrestrictedStockQuantity) AS on_hand_qty,
  SUM(DailyReceiptQuantity)      AS receipt_qty,
  SUM(DailyIssueQuantity)        AS issue_qty,
  SUM(DailyNetMovement)          AS net_movement_qty,
  SUM(TrailingDailyConsumptionRate) AS daily_consumption_rate,
  CASE WHEN SUM(TrailingDailyConsumptionRate) > 0
       THEN SUM(UnrestrictedStockQuantity) / SUM(TrailingDailyConsumptionRate) END AS days_of_cover,
  MAX(UoM_MEINS) AS uom
FROM {SNAP}
GROUP BY 1,2,3,4
""")
timeseries = (timeseries
    .withColumn("refresh_id", F.lit(REFRESH_ID))
    .withColumn("generated_at", F.current_timestamp()))
(timeseries.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TS_OUT))
spark.sql(f"COMMENT ON TABLE {TS_OUT} IS 'Inventory twin metric time-series per material x plant x date: on-hand, receipts, issues, net movement, days-of-cover.'")
print(f"Wrote {timeseries.count():,} rows to {TS_OUT}")

## Column descriptions
Apply human-readable descriptions to every output column (persisted on rebuild).

In [ ]:
# Column descriptions — re-applied after the overwrite writes so they persist in the
# catalog on every rebuild. EXTEND for your business.
COLUMN_COMMENTS = {
    CUR_OUT: {
        "mandt": "SAP client identifier",
        "material": "Material / SKU number",
        "plant": "Plant",
        "material_text": "Material description",
        "material_group": "Material group / category",
        "uom": "Unit of measure",
        "snapshot_date": "Date of the latest inventory snapshot used",
        "on_hand_qty": "Unrestricted on-hand stock across storage locations and batches",
        "daily_consumption_rate": "Trailing daily consumption rate",
        "days_of_cover_actual": "Days of cover using the trailing consumption rate",
        "demand_13w_avg_wk": "Weekly demand from the Demand twin (summed across channels)",
        "next_period_forecast_qty": "Next-period demand forecast (from the Demand twin)",
        "horizon_total_forecast_qty": "Total forecast demand over the horizon (from the Demand twin)",
        "daily_demand": "Estimated daily demand derived from the 13-week weekly average",
        "days_of_cover_demand": "Days of cover using forecast / recent demand",
        "horizon_coverage_ratio": "On-hand divided by total horizon forecast demand",
        "is_stockout": "True when on-hand stock is at or below zero",
        "is_low_cover": "True when demand-based days of cover is below the low-cover threshold",
        "is_overstock": "True when demand-based days of cover exceeds the overstock threshold",
        "projected_stockout_date": "Estimated date stock runs out at current demand",
        "as_of_date": "As-of date of this twin snapshot",
        "refresh_id": "Unique id of the refresh run",
        "generated_at": "Timestamp this row was generated",
    },
    TS_OUT: {
        "mandt": "SAP client identifier",
        "material": "Material / SKU number",
        "plant": "Plant",
        "period_start": "Snapshot date",
        "on_hand_qty": "Unrestricted on-hand stock at the snapshot date",
        "receipt_qty": "Quantity received during the period",
        "issue_qty": "Quantity issued / consumed during the period",
        "net_movement_qty": "Net stock movement during the period",
        "daily_consumption_rate": "Trailing daily consumption rate at the snapshot",
        "days_of_cover": "Days of cover at the snapshot",
        "uom": "Unit of measure",
        "refresh_id": "Unique id of the refresh run",
        "generated_at": "Timestamp this row was generated",
    },
}
for _tbl, _cols in COLUMN_COMMENTS.items():
    for _c, _desc in _cols.items():
        spark.sql(f"ALTER TABLE {_tbl} ALTER COLUMN {_c} COMMENT '{_desc}'")
print(f"Applied column descriptions to {CUR_OUT} and {TS_OUT}")

## 3. Verify

In [ ]:
print("=== inventory_twin_current (most at-risk) ===")
display(spark.sql(f"""
SELECT material, plant, material_group, on_hand_qty, ROUND(days_of_cover_demand,1) doc_demand,
       is_stockout, is_low_cover, is_overstock, projected_stockout_date, horizon_total_forecast_qty
FROM {CUR_OUT} ORDER BY days_of_cover_demand ASC NULLS FIRST LIMIT 25
"""))
print("=== risk summary ===")
display(spark.sql(f"SELECT SUM(CAST(is_stockout AS INT)) stockouts, SUM(CAST(is_low_cover AS INT)) low_cover, SUM(CAST(is_overstock AS INT)) overstock, COUNT(*) positions FROM {CUR_OUT}"))
print("=== timeseries span ===")
display(spark.sql(f"SELECT COUNT(*) rows, COUNT(DISTINCT material||plant) series, MIN(period_start) first_p, MAX(period_start) last_p FROM {TS_OUT}"))

## Next
Inventory twin live. Next: **Procurement** (`05_procurement_twin`) — inbound supply,
supplier OTIF/lead-time/reliability, open POs and expected receipts.